# PW9 - Classification with Support Vector Machines (SVM)

**Team:** Noé Berdoz, Michael Strefeler

Our starting levels, the time spent per exercise and the bonus request are in the preamble of `ex1-digit-classification-stud.ipynb`.

# Ex2 - Review questions

### a) What are the two fundamental ideas a SVM are built on ? Summarize them with your own words.

The first idea is to maximise the margin. When two classes can be separated, there are many borders that classify all the training points correctly, and logistic regression just stops at the first one it finds. An SVM looks for the border that is as far as possible from the closest points of each class. These closest points are called the support vectors. A border with a large margin is safer for new points that fall between the classes. In the loss function, this is done by minimising $\|w\|^2/2$, because the margin is equal to $2 / \|w\|$.

The second idea is the kernel trick. When the classes cannot be separated by a straight border, we would normally have to try many extra features by hand, like $x_1^2$ or $x_1 x_2$, as we did with logistic regression. With a kernel, the SVM maps the points into a space with more dimensions, where a linear border can separate them. We only need to choose a kernel, such as polynomial or RBF, and the SVM does the rest without computing the new features directly.

### b) With the hinge loss, training points can fall into three cases. Re-explain these cases with your own words.

In the first case, the point is on the correct side of the border and outside the margin. The model is confident about this point, so its hinge loss is 0 and it does not move the border.

In the second case, the point is on the correct side of the border but inside the margin. It is classified correctly, but the model is not very confident. Its hinge loss is between 0 and 1. It is 0 when the point is exactly on the edge of the margin and 1 when it is exactly on the border.

In the third case, the point is on the wrong side of the border, so it is misclassified. Its hinge loss is larger than 1 and grows with the distance to the border, so points that are very far on the wrong side have a large penalty.

### c) What are the two implementations of SVMs available in SciKit Learn ? Which one would you take if you have a system that needs to incorporate incremental learning ?

The first implementation is `svm.SVC()`, which is based on the libsvm library. It solves the minimisation problem with constraints using numerical methods. It is well optimised and stable, it needs little tuning, and it supports kernels.

The second implementation is `linear_model.SGDClassifier()` with `loss='hinge'` and `penalty='l2'`. It trains a linear SVM with gradient descent on the hinge loss plus the regularisation term. Instead of C, it uses the parameter alpha, which gives weight to the regularisation term. We also need to tune the learning rate.

For a system that needs incremental learning, we would choose `SGDClassifier()`. With gradient descent, the model can be updated with new samples (for example with `partial_fit()`), without training again on the whole dataset. It also works better than `SVC()` on very large training sets.

### d) A SVM can classify between 2 classes. Cite and explain in your own words the 2 strategies we have to build a multi-class (with K classes) system with SVM ?

The first strategy is one-vs-all. We train K SVMs, one per class. Each SVM learns to separate its class from all the other classes together. To classify a new sample, we give it to the K SVMs, and the SVM with the highest output value wins and gives its class to the sample.

The second strategy is one-vs-one. We train one SVM for each pair of classes, so $K(K-1)/2$ SVMs in total. Each SVM only learns to separate two classes. To classify a new sample, every SVM votes for one of its two classes, and the class with the most votes is given to the sample.

### e) Are the strategies of point d) equal in terms of cpu ? (elaborate your answer considering training and testing times)

No, they are not equal.

For training, one-vs-all trains only K SVMs, but each of them uses all N training samples. One-vs-one trains many more SVMs, $K(K-1)/2$, but each of them only uses the samples of two classes, so about $2N/K$ samples when the classes are balanced. The training time of an SVM grows faster than the number of samples, so many small problems are often faster than a few large ones. Because of this, one-vs-one is usually faster to train, even if it has more SVMs.

For testing, one-vs-all needs to compute K SVM outputs for each new sample, while one-vs-one needs $K(K-1)/2$ outputs. This difference gets bigger when K grows. For our 10 digits in exercise 1, this means 10 SVMs against 45 SVMs, so one-vs-one is usually slower to test.

### f) Describe a machine learning task for which SVM would be a better choice than any of the algorithms previously studied. Explain why.

A good example is the recognition of handwritten digits, like in exercise 1. We have a few thousand training images, each with 784 numerical features, and the borders between the classes are not linear.

The Bayesian classifier would need to model the distribution of 784 features in each class, which needs a lot of data or strong hypotheses like naive Bayes, where the pixels are assumed to be independent, which is clearly false for images. The k-NN classifier has no training, but it has to compare each new image with all the training images, which is slow at test time, and in 784 dimensions the distances become less meaningful. Logistic regression only gives a linear border, unless we add extra features by hand, which is not realistic with 784 features.

An SVM solves these problems. The RBF or polynomial kernel gives non-linear borders without having to create the features by hand, and maximising the margin helps the model to generalise well to new images. At test time, the SVM only needs the support vectors, not all the training images. In exercise 1, our RBF SVM reached a test accuracy of 0.925 on raw pixels with only 2000 training images.